# ATC Multi-Agent GRPO Training — Jupyter Server

Mirrors `alurm_runner.sbatch` exactly. Same pinned deps, same training call, same plot output.

**Run order:** top to bottom. Edit the **Config** cell first.

## 0. Config — edit before running

In [ ]:
from pathlib import Path
import os

# ── Paths ──────────────────────────────────────────────────────────────────
# Set REPO_DIR to wherever you cloned the repo on this server.
REPO_DIR   = Path(".").resolve()          # assumes notebook lives inside repo
OUTPUT_DIR = Path("/tmp/atc/outputs")     # change to a persistent path if needed
LOGS_DIR   = Path("/tmp/atc/logs")

# ── Model & training ───────────────────────────────────────────────────────
SMOKE_MODEL  = "Qwen/Qwen2.5-1.5B-Instruct"   # fast sanity-check model
TRAIN_MODEL  = "Qwen/Qwen2.5-7B-Instruct"      # full training model
EPISODES     = 2          # increase for real run (200 for full)
N_GENERATIONS = 4
SEED         = 42
SKIP_SMOKE   = False      # set True to skip the 1-episode sanity run

# ── W&B (optional) ─────────────────────────────────────────────────────────
# Leave blank to run offline. Or set key directly here.
WANDB_KEY    = ""         # e.g. "abcdef123..."  (overrides .env)
WANDB_PROJECT = "atc-multiagent-grpo"

# ── Derived ────────────────────────────────────────────────────────────────
SMOKE_OUTPUT_DIR = OUTPUT_DIR / "atc-smoke"
TRAIN_OUTPUT_DIR = OUTPUT_DIR / "atc-multiagent"
PLOTS_DIR        = OUTPUT_DIR / "plots"

for d in (OUTPUT_DIR, LOGS_DIR, SMOKE_OUTPUT_DIR, TRAIN_OUTPUT_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"REPO_DIR  : {REPO_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"PLOTS_DIR : {PLOTS_DIR}")

## 1. Environment variables

In [ ]:
import os, socket, time

os.environ["MASTER_ADDR"]                = "127.0.0.1"
os.environ["MASTER_PORT"]                = str(30000 + (os.getpid() % 2000))
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["PIP_NO_CACHE_DIR"]           = "1"
os.environ["NCCL_DEBUG"]                 = "INFO"
os.environ["NCCL_IB_DISABLE"]            = "1"
os.environ["NCCL_P2P_DISABLE"]           = "0"
os.environ["OMP_NUM_THREADS"]            = "4"
os.environ["MKL_NUM_THREADS"]            = "4"
os.environ["PYTHONUNBUFFERED"]           = "1"
# CRITICAL: disable torch.compile to avoid Dynamo errors with GRPO
os.environ["TORCH_COMPILE_DISABLE"]      = "1"
os.environ["WORLD_SIZE"]                 = "1"
os.environ["RANK"]                       = "0"
os.environ["LOCAL_RANK"]                 = "0"
os.environ["LOCAL_WORLD_SIZE"]           = "1"
os.environ["PYTHONPATH"]                 = str(REPO_DIR) + ":" + os.environ.get("PYTHONPATH", "")

print(f"Hostname       : {socket.gethostname()}")
print(f"Start time     : {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"MASTER_PORT    : {os.environ['MASTER_PORT']}")
print(f"TORCH_COMPILE_DISABLE=1  (Dynamo disabled)")

## 2. GPU check

In [ ]:
import subprocess, sys

subprocess.run(["nvidia-smi"], check=False)
print(f"Python: {sys.version}")

## 3. Load W&B key from .env (if present)

In [ ]:
import re

def _load_dotenv(path):
    """Minimal .env loader — no external deps needed."""
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        m = re.match(r'^(?:export\s+)?([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(.*)$', line)
        if m:
            k, v = m.group(1), m.group(2).strip().strip('"\'')
            if k not in os.environ:   # don't overwrite already-set vars
                os.environ[k] = v

for env_file in (REPO_DIR / ".env", REPO_DIR / ".env.wandb"):
    _load_dotenv(env_file)
    if env_file.exists():
        print(f"Loaded: {env_file}")

# Explicit key in config cell takes priority
if WANDB_KEY.strip():
    os.environ["WANDB_API_KEY"] = WANDB_KEY.strip()

# Normalize key: strip assignment prefix + whitespace (common .env mistake)
raw_key = os.environ.get("WANDB_API_KEY", "")
raw_key = raw_key.lstrip("\r").split("=")[-1].strip()
if raw_key:
    os.environ["WANDB_API_KEY"]  = raw_key
    os.environ["WANDB_MODE"]     = "online"
    os.environ["WANDB_PROJECT"]  = WANDB_PROJECT
    print(f"W&B key found (len={len(raw_key)}) → mode=online")
else:
    os.environ.pop("WANDB_API_KEY", None)
    os.environ["WANDB_MODE"] = "offline"
    print("W&B offline (no key)")

## 4. Clear stale Unsloth pycache

In [ ]:
import shutil

removed = 0
for p in REPO_DIR.rglob("__pycache__"):
    try:
        shutil.rmtree(p)
        removed += 1
    except Exception:
        pass
print(f"Cleared {removed} __pycache__ dirs")

## 5. Install pinned packages

**Same versions as `alurm_runner.sbatch`.** Do not change these.

In [ ]:
import subprocess, sys

def pip(*args):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    result = subprocess.run(cmd, capture_output=False)
    if result.returncode != 0:
        raise RuntimeError(f"pip failed: {' '.join(args[:4])}")

print("Removing vllm (conflicts with unsloth)...")
pip("uninstall", "-y", "vllm", "vllm-flash-attn")

print("\nInstalling training deps (pinned, --no-deps)...")
pip(
    "install", "--upgrade", "--no-input", "--no-deps",
    "trl==0.16.0",
    "unsloth==2026.4.7",
    "unsloth-zoo==2026.4.9",
    "accelerate==1.13.0",
    "peft==0.19.1",
    "bitsandbytes==0.49.2",
    "xformers==0.0.31",
    "datasets==2.20.0",
)

print("\nInstalling utility deps (pinned, --no-deps)...")
pip(
    "install", "--upgrade", "--no-input", "--no-deps",
    "multiprocess==0.70.17",
    "pyarrow-hotfix==0.7",
    "xxhash==3.6.0",
    "huggingface-hub==0.36.2",
    "hf_transfer==0.1.9",
    "tyro==0.9.17",
)

print("\nInstalling wandb...")
pip("install", "--upgrade", "--no-input", "wandb>=0.18,<0.20")

print("\nDone.")

## 6. Verify installations

In [ ]:
import importlib

# Force-reimport after pip install in same process
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ("unsloth", "trl", "peft", "accelerate", "bitsandbytes")):
        del sys.modules[mod]

import torch
import transformers
import trl
import unsloth
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel

print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"TRL          : {trl.__version__}")
print(f"Unsloth      : {unsloth.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("All imports OK")

## 7. W&B login

In [ ]:
if os.environ.get("WANDB_MODE") == "online":
    try:
        import wandb
        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
        print(f"W&B logged in  project={WANDB_PROJECT}")
    except Exception as exc:
        print(f"W&B login failed ({exc}) → switching to offline")
        os.environ["WANDB_MODE"] = "offline"
else:
    print("W&B offline")

## 8. Smoke gun — 1-episode sanity run

Fast check that the full stack works before committing to a long run.  
Set `SKIP_SMOKE = True` in the Config cell to skip.

In [ ]:
if not SKIP_SMOKE:
    print(f"===== SMOKE GUN START (model={SMOKE_MODEL}, episodes=1) =====")
    result = subprocess.run(
        [
            sys.executable, str(REPO_DIR / "training" / "train_grpo.py"),
            "--model",        SMOKE_MODEL,
            "--output_dir",   str(SMOKE_OUTPUT_DIR),
            "--episodes",     "1",
            "--n_generations","2",
            "--seed",         str(SEED),
            "--no_eval",
        ],
        env=os.environ,
        cwd=str(REPO_DIR),
    )
    if result.returncode != 0:
        raise RuntimeError(f"Smoke gun FAILED (exit {result.returncode}) — fix before full run")
    print("===== SMOKE GUN COMPLETE =====")
else:
    print("Smoke gun skipped (SKIP_SMOKE=True)")

## 9. Main training run

In [ ]:
import time

print(f"===== START TRAINING =====")
print(f"Model      : {TRAIN_MODEL}")
print(f"Episodes   : {EPISODES}")
print(f"Generations: {N_GENERATIONS}")
print(f"Output     : {TRAIN_OUTPUT_DIR}")
print()

t0 = time.monotonic()
result = subprocess.run(
    [
        sys.executable, str(REPO_DIR / "training" / "train_grpo.py"),
        "--model",         TRAIN_MODEL,
        "--output_dir",    str(TRAIN_OUTPUT_DIR),
        "--episodes",      str(EPISODES),
        "--n_generations", str(N_GENERATIONS),
        "--seed",          str(SEED),
    ],
    env=os.environ,
    cwd=str(REPO_DIR),
)
elapsed = time.monotonic() - t0

if result.returncode != 0:
    raise RuntimeError(f"Training FAILED (exit {result.returncode})")
print(f"===== TRAINING COMPLETE ({elapsed/60:.1f} min) =====")

## 10. Generate plots

In [ ]:
import json
import sys

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import matplotlib
matplotlib.use("Agg")
from training.plot_rewards import plot_training_curves, plot_eval_comparison

generated = []

# ── Training curves ────────────────────────────────────────────────────────
curves_path = TRAIN_OUTPUT_DIR / "reward_curves.json"
if curves_path.exists():
    data = json.loads(curves_path.read_text())
    plot_training_curves(data, save_dir=str(PLOTS_DIR), show=False)
    generated.append(PLOTS_DIR / "training_curves.png")
    print(f"Saved: training_curves.png")
else:
    print(f"[WARN] {curves_path} not found")

# ── Before / after comparison ──────────────────────────────────────────────
base_path    = TRAIN_OUTPUT_DIR / "base_model_metrics.json"
trained_path = TRAIN_OUTPUT_DIR / "trained_model_metrics.json"
if base_path.exists() and trained_path.exists():
    eval_data = {
        "base":    json.loads(base_path.read_text()),
        "trained": json.loads(trained_path.read_text()),
    }
    plot_eval_comparison(eval_data, save_dir=str(PLOTS_DIR), show=False)
    generated.append(PLOTS_DIR / "eval_comparison.png")
    print(f"Saved: eval_comparison.png")
else:
    print(f"[WARN] base/trained metrics not found — skipping eval_comparison")

print(f"\nAll plots → {PLOTS_DIR}")

## 11. Display plots

In [ ]:
from IPython.display import Image, display

for png in sorted(PLOTS_DIR.glob("*.png")):
    print(f"── {png.name} ──")
    display(Image(filename=str(png), width=900))

## 12. Output file summary

In [ ]:
print(f"Output directory: {TRAIN_OUTPUT_DIR}")
print()
for f in sorted(TRAIN_OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        unit = "KB" if size < 1_000_000 else "MB"
        val  = size / 1_000 if size < 1_000_000 else size / 1_000_000
        print(f"  {f.relative_to(TRAIN_OUTPUT_DIR):<50}  {val:6.1f} {unit}")

print()
print(f"Plots directory: {PLOTS_DIR}")
for f in sorted(PLOTS_DIR.glob("*.png")):
    print(f"  {f.name}")